# CrackSAM Benchmark: Unsupervised Generalized Frangi Graph vs. Supervised CrackSAM
## Evaluation on the VT-GraF-Dataset (Visible-Thermal Pavement Fissures)

This Colab Pro notebook reproduces the evaluation on the **VT-GraF-Dataset** comparing:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised, training-free multi-modal graph-based method (using MST + Weighted Betweenness Centrality on GPU, weights: 1/3 Visible + 2/3 Infrared).
2. **CrackSAM (Adapter)**: The Segment Anything foundation model (ViT-H backbone) fine-tuned on the Khanhha dataset using an **Adapter** PEFT method.
3. **CrackSAM (LoRA)**: SAM ViT-H fine-tuned using **LoRA** (Low-Rank Adaptation).
4. **CrackSAM (Adapter + LoRA)**: SAM ViT-H fine-tuned using **both** PEFT methods.

---  
### Technical requirements:
- Select a **GPU runtime** (T4, V100, or A100) in Colab Pro.
- Ensure your Google Drive is mounted to save/retrieve checkpoints if necessary, or the repository files are uploaded.

### Connect Google Drive
Mount your Drive to easily load/save files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Environment Setup & Dependencies
We install the standard libraries and retrieve the pre-trained SAM ViT-H weights.

In [ ]:
import subprocess
import sys
import os

print("Installing standard packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "POT", "scikit-image", "opencv-python", "matplotlib", "scipy", "torch", "torchvision", "pypdf"])

print("Creating checkpoints directory...")
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/sam_vit_h_4b8939.pth'):
    print("Downloading base SAM ViT-H weights (sam_vit_h_4b8939.pth)...")
    subprocess.run(["wget", "-q", "-O", "checkpoints/sam_vit_h_4b8939.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"], check=True)
    print("Base SAM weights downloaded.")
else:
    print("Base SAM weights already present.")

print("Setup complete!")

### Download the VT-GraF-Dataset
We download the aligned visible and thermal dataset (5 fissures) from Google Drive if it isn't already present.

In [ ]:
import os
import gdown
from pathlib import Path

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'VT-GraF-Dataset'

def check_dataset_exists():
    known_dirs = [
        Path('VT-GraF-Dataset'),
        Path('VT-GraF'),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/VT-GraF-Dataset'),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/VT-GraF'),
        Path('/content/drive/MyDrive/Datasets/Raphael'),
        Path('/content/drive/MyDrive/Datasets/Raphael/VT-GraF-Dataset')
    ]
    for kd in known_dirs:
        if (kd / 'Fissure 1').exists():
            return True
            
    def scan_dir(p):
        try:
            for entry in os.scandir(p):
                if entry.name.startswith('.'):
                    continue
                if entry.is_dir():
                    if entry.name == 'Fissure 1':
                        return True
                    if scan_dir(entry.path):
                        return True
        except Exception:
            pass
        return False
        
    return scan_dir('.')

if not check_dataset_exists():
    print("Downloading VT-GraF-Dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download completed.")
else:
    print("Dataset already present.")

## 2. Load the Generalized Frangi Graph & CrackSAM Modules
We locate the directories containing the source code in your workspace and import the modules.

In [ ]:
import sys
import os
import shutil
import subprocess
from pathlib import Path

def find_isprs_src():
    paths_to_check = [
        Path('ISPRS'),
        Path('../ISPRS'),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/Datasets/Raphael/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/Datasets/Raphael/ISPRS'),
        Path('/content/drive/MyDrive/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/ISPRS')
    ]
    for p in paths_to_check:
        if p.exists() and (p / 'src').exists():
            return p.resolve()
            
    # Scan Drive if mounted
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        try:
            for child in drive_root.iterdir():
                if child.is_dir():
                    if child.name == 'ISPRS' and (child / 'src').exists():
                        return child.resolve()
                    potential = child / 'ISPRS'
                    if potential.exists() and (potential / 'src').exists():
                        return potential.resolve()
                    potential2 = child / 'Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset' / 'ISPRS'
                    if potential2.exists() and (potential2 / 'src').exists():
                        return potential2.resolve()
        except Exception:
            pass
    return None

isprs_path = find_isprs_src()

if isprs_path is None:
    print("Cloning repository to access code modules...")
    try:
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "https://github.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset.git"
        ], check=True, timeout=30)
        subprocess.run([
            "git", "-C", "Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset",
            "sparse-checkout", "set", "ISPRS"
        ], check=True)
        isprs_path = Path("Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS").resolve()
    except Exception as e:
        raise RuntimeError("Failed to clone repository. Please upload the 'ISPRS' folder manually to Colab.")

sys.path.insert(0, str(isprs_path))
from src import VTGraFDataset, extract_frangi_graph_gpu, skeletonize_lee, thicken, compute_metrics, wasserstein_distance_skeletons
print(f"Successfully loaded ISPRS modules from {isprs_path}")

### Load CrackSAM Code base & Checkpoints
We locate the `CrackSAM/CrackSAM` codebase and checkpoints. We copy the `.pth` files to our local environment and insert the path into `sys.path`.

In [ ]:
def find_cracksam_src():
    paths_to_check = [
        Path('ISPRS/CrackSAM/CrackSAM/CrackSAM'),
        Path('../ISPRS/CrackSAM/CrackSAM/CrackSAM'),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS/CrackSAM/CrackSAM/CrackSAM'),
        Path('/content/drive/MyDrive/Datasets/Raphael/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS/CrackSAM/CrackSAM/CrackSAM'),
        Path('/content/drive/MyDrive/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS/CrackSAM/CrackSAM/CrackSAM')
    ]
    for p in paths_to_check:
        if p.exists() and (p / 'delta').exists():
            return p.resolve()
            
    # Scan Drive
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        try:
            for child in drive_root.iterdir():
                if child.is_dir():
                    potential = child / 'ISPRS' / 'CrackSAM' / 'CrackSAM' / 'CrackSAM'
                    if potential.exists() and (potential / 'delta').exists():
                        return potential.resolve()
                    potential2 = child / 'Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset' / 'ISPRS' / 'CrackSAM' / 'CrackSAM' / 'CrackSAM'
                    if potential2.exists() and (potential2 / 'delta').exists():
                        return potential2.resolve()
        except Exception:
            pass
            
    # Scan current folder
    for child in Path('.').iterdir():
        if child.is_dir() and 'Generalized-Frangi' in child.name:
            potential = child / 'ISPRS' / 'CrackSAM' / 'CrackSAM' / 'CrackSAM'
            if potential.exists() and (potential / 'delta').exists():
                return potential.resolve()
    return None

cracksam_path = find_cracksam_src()

if cracksam_path is None:
    print("Cloning repository to retrieve CrackSAM code...")
    try:
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/KG-TSI-Civil/CrackSAM.git"
        ], check=True, timeout=30)
        cracksam_path = Path("CrackSAM/CrackSAM").resolve()
    except Exception as e:
        raise RuntimeError("Failed to load CrackSAM codebase. Please upload it manually.")

sys.path.insert(0, str(cracksam_path))
print(f"Successfully loaded CrackSAM path: {cracksam_path}")

# Search and copy fine-tuned checkpoints
delta_ckpts = {
    'adapter': 'CrackSAM_adapter_d32.pth',
    'lora': 'CrackSAM_lora_qv_r4.pth',
    'both': 'CrackSAM_adapter_lora_d32_r8.pth'
}

for dtype, fname in delta_ckpts.items():
    local_path = Path('checkpoints') / fname
    if local_path.exists():
        print(f"Checkpoint {fname} already in local checkpoints folder.")
        continue
        
    # Find file in workspace
    source_file = None
    potential_source_paths = [
        cracksam_path / 'checkpoints' / fname,
        cracksam_path.parent / 'checkpoints' / fname,
        cracksam_path.parent.parent / 'checkpoints' / fname,
        Path('/content/drive/MyDrive/Datasets/Raphael/checkpoints') / fname,
        Path('/content/drive/MyDrive/checkpoints') / fname
    ]
    for ps in potential_source_paths:
        if ps.exists():
            source_file = ps
            break
            
    if source_file is not None:
        shutil.copy(str(source_file), str(local_path))
        print(f"Copied {fname} from {source_file.parent} to checkpoints/")
    else:
        print(f"WARNING: Could not find checkpoint {fname} locally. Please verify your Drive mount or workspace files.")

## 3. Load and Initialize CrackSAM Models
We load the base Segment Anything weights, register the fine-tuning Adapter/LoRA deltas, and initialize the models on GPU.

In [ ]:
import torch
from importlib import import_module
from segment_anything import sam_model_registry

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset = VTGraFDataset('.')
print(f"Dataset loaded with {len(dataset)} fissures.")

def load_cracksam(delta_type, delta_ckpt, vit_name='vit_h', checkpoint='checkpoints/sam_vit_h_4b8939.pth', img_size=448, num_classes=1, middle_dim=32, scaling_factor=0.2, rank=4):
    print(f"Building base SAM ({vit_name})...")
    sam, img_embedding_size = sam_model_registry[vit_name](
        image_size=img_size,
        num_classes=num_classes,
        checkpoint=checkpoint,
        pixel_mean=[0, 0, 0],
        pixel_std=[1, 1, 1]
    )
    
    print(f"Wrapping with {delta_type} modules...")
    if delta_type == 'adapter':
        pkg = import_module('delta.sam_adapter_image_encoder')
        net = pkg.Adapter_Sam(sam, middle_dim, scaling_factor)
    elif delta_type == 'lora':
        pkg = import_module('delta.sam_lora_image_encoder')
        net = pkg.LoRA_Sam(sam, rank)
    elif delta_type == 'both':
        pkg = import_module('delta.sam_adapter_lora_image_encoder')
        net = pkg.LoRA_Adapter_Sam(sam, middle_dim, rank)
        
    net = net.to(device)
    print(f"Loading delta parameters from {delta_ckpt}...")
    net.load_delta_parameters(delta_ckpt)
    net.eval()
    return net

# Try loading the three models
models = {}
for dtype, fname in delta_ckpts.items():
    cpath = Path('checkpoints') / fname
    if cpath.exists():
        try:
            models[dtype] = load_cracksam(
                delta_type=dtype,
                delta_ckpt=str(cpath),
                checkpoint='checkpoints/sam_vit_h_4b8939.pth',
                rank=4 if dtype == 'lora' else 8
            )
            print(f"Model {dtype} loaded successfully!")
        except Exception as e:
            print(f"Error loading model {dtype}: {e}")
    else:
        print(f"Skipping {dtype} (checkpoint file missing).")

## 4. Define Inference Pipeline for CrackSAM
CrackSAM is a visible-only model that operates on a 3-channel RGB image resized to $448 \times 448$ with pixel values scaled in $[0, 1]$.

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import zoom

def run_cracksam_inference(net, sample, patch_size=448):
    # Get visible grayscale image [0, 1] as numpy
    img_vis_np = (sample['visible'].numpy() * 255).astype(np.uint8)
    h_orig, w_orig = img_vis_np.shape
    
    # Convert to 3-channel RGB
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    
    # Resize to patch_size (448, 448) and scale to [0, 1]
    img_vis_resized = cv2.resize(img_vis_rgb, (patch_size, patch_size), interpolation=cv2.INTER_CUBIC)
    img_vis_resized = img_vis_resized.astype(np.float32) / 255.0
    
    # Permute channels to shape (3, 448, 448) and add batch dimension
    inputs = torch.from_numpy(np.transpose(img_vis_resized, (2, 0, 1))).float().unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Multimask output is False for binary crack segmentation
        outputs = net(inputs, False, patch_size)
        output_masks = outputs['masks']  # shape: (1, 2, 448, 448)
        
        # Extract binary segmentation prediction via softmax + argmax
        out = torch.argmax(torch.softmax(output_masks, dim=1), dim=1).squeeze(0)  # shape: (448, 448)
        prediction = out.cpu().detach().numpy().astype(np.uint8)
        
    # Resize back to original size via nearest-neighbor interpolation to preserve binary edges
    prediction_orig = cv2.resize(prediction, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
    
    # Extract skeleton centerline
    skel = skeletonize_lee(prediction_orig)
    return prediction_orig, skel

## 5. Quantitative Evaluation and Benchmarking
We run the evaluation across all 5 fissures of the **VT-GraF-Dataset**.
- **Ours (Generalized Frangi Graph)** uses scale set $\Sigma=[30, 40, 50]$, radius $R=3$, and triangle connectivity $K=2$ to handle asphalt clutter.
- **CrackSAM** predicts on the visible modality.
- Skeletons are thickened to **5 pixels** for overlap metrics evaluation.

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

results = []
visualizations = {}
weights_ours = {'visible': 1/3, 'infrared': 2/3}
sigmas_benchmark = [30, 40, 50]

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OUR METHOD (Generalized Frangi Graph, K=2) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=sigmas_benchmark, R=3, K=2, device=device
    )
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    sk_ours = skeletonize_lee(pred_ours)
    sk_thick_ours = thicken(sk_ours, pixels=5)
    
    # --- 2. EVALUATE CRACKSAM MODELS ---
    preds_cracksam = {}
    sk_thick_cracksam = {}
    
    for mname, model in models.items():
        pred_cs, sk_cs = run_cracksam_inference(model, sample)
        sk_thick_cs = thicken(sk_cs, pixels=5)
        preds_cracksam[mname] = pred_cs
        sk_thick_cracksam[mname] = sk_thick_cs
        
    # --- 3. GROUND TRUTH ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_thick_ours, sk_gt_thick)
    
    res_dict = {
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours
    }
    
    for mname in delta_ckpts.keys():
        if mname in sk_thick_cracksam:
            jac_cs, tv_cs = compute_metrics(sk_thick_cracksam[mname], sk_gt_thick)
            wass_cs = wasserstein_distance_skeletons(sk_thick_cracksam[mname], sk_gt_thick)
            res_dict[f'CrackSAM_{mname}_Jaccard'] = jac_cs
            res_dict[f'CrackSAM_{mname}_Tversky'] = tv_cs
            res_dict[f'CrackSAM_{mname}_Wasserstein'] = wass_cs
        else:
            res_dict[f'CrackSAM_{mname}_Jaccard'] = np.nan
            res_dict[f'CrackSAM_{mname}_Tversky'] = np.nan
            res_dict[f'CrackSAM_{mname}_Wasserstein'] = np.nan
            
    results.append(res_dict)
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours': sk_thick_ours,
        'ours_similarity': sim_img,
        'ours_centrality': centrality_i,
        'cracksam_preds': preds_cracksam,
        'cracksam_skels': sk_thick_cracksam
    }

df_res = pd.DataFrame(results)

## 6. Comparison Results Table
We present the quantitative results and show the global averages across all 5 fissures.

In [ ]:
pd.set_option('display.precision', 4)
display(df_res)

print("\n" + "="*70)
print("--- GLOBAL BENCHMARK AVERAGES ---")
print("="*70)
print(f"Jaccard (IoU): Ours = {df_res['Ours_Jaccard'].mean():.4f} | CS_Adapter = {df_res['CrackSAM_adapter_Jaccard'].mean():.4f} | CS_LoRA = {df_res['CrackSAM_lora_Jaccard'].mean():.4f} | CS_Both = {df_res['CrackSAM_both_Jaccard'].mean():.4f}")
print(f"Tversky Index: Ours = {df_res['Ours_Tversky'].mean():.4f} | CS_Adapter = {df_res['CrackSAM_adapter_Tversky'].mean():.4f} | CS_LoRA = {df_res['CrackSAM_lora_Tversky'].mean():.4f} | CS_Both = {df_res['CrackSAM_both_Tversky'].mean():.4f}")
print(f"Wasserstein  : Ours = {df_res['Ours_Wasserstein'].mean():.4f} | CS_Adapter = {df_res['CrackSAM_adapter_Wasserstein'].mean():.4f} | CS_LoRA = {df_res['CrackSAM_lora_Wasserstein'].mean():.4f} | CS_Both = {df_res['CrackSAM_both_Wasserstein'].mean():.4f}")
print("="*70)

## 7. Visual Inspection
We plot each processing step side-by-side to visually inspect the qualitative output.

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs('results_images_cracksam', exist_ok=True)

for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    h, w = vis_data['visible'].shape
    
    # We create a 2x3 grid to plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Row 0: Inputs and GT
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 0].set_title("Visible Modality")
    
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 1].set_title("Inverted Thermal")
    
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    axes[0, 2].set_title("Ground Truth")
    
    # Row 1: Predictions (Ours vs CrackSAM versions)
    axes[1, 0].imshow(vis_data['ours'], cmap='hot')
    axes[1, 0].set_title("Ours: Generalized Frangi Graph (K=2)")
    
    # Display Adapter or LoRA prediction
    if 'lora' in vis_data['cracksam_skels']:
        axes[1, 1].imshow(vis_data['cracksam_skels']['lora'], cmap='hot')
        axes[1, 1].set_title("CrackSAM (LoRA)")
    else:
        axes[1, 1].imshow(np.zeros_like(vis_data['visible']), cmap='gray')
        axes[1, 1].set_title("CrackSAM (LoRA) Missing")
        
    # Overlay
    rgba_overlay = np.zeros((h, w, 4))
    rgba_overlay[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4] # White = GT
    rgba_overlay[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green = Ours
    if 'lora' in vis_data['cracksam_skels']:
        rgba_overlay[vis_data['cracksam_skels']['lora'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red = CrackSAM
        axes[1, 2].set_title("Overlay (White:GT, Green:Ours, Red:LoRA)")
    else:
        axes[1, 2].set_title("Overlay (White:GT, Green:Ours)")
        
    axes[1, 2].imshow(vis_data['visible'], cmap='gray')
    axes[1, 2].imshow(rgba_overlay)
    
    for ax in axes.ravel():
        ax.axis('off')
        
    plt.tight_layout()
    plt.savefig(f'results_images_cracksam/{name}_comparison.png', dpi=150)
    plt.show()